# High-Volume ADE (DPT-3) → Snowflake — Demo

Parse and extract a batch of invoices with **LandingAI ADE DPT-3**, and stream the
structured results into **Snowflake** via staged `COPY INTO`.

Run the cells top to bottom. Watch **step 5** — the progress bar and the run
summary show the speed.

## 1  ·  Configuration

In [ ]:
from config import Settings

S = Settings()
print('✅ Configuration loaded')
print(f'   Parse model    : {S.PARSE_MODEL}')
print(f'   Extract model  : {S.EXTRACT_MODEL}')
print(f'   Workers        : {S.MAX_WORKERS}')
print(f'   Snowflake target: {S.DATABASE}.{S.SNOWFLAKE_SCHEMA}')

## 2  ·  The input documents

A folder of invoices — different vendors, different layouts, no templates.

In [ ]:
from run_demo import gather_files

files = gather_files(S.input_dir, S.file_exts)
print(f'📄 Found {len(files)} document(s) in "{S.input_dir}":')
for f in files:
    print('   •', f.split('/')[-1])

## 3  ·  Canary — parse + extract one invoice

DPT-3 runs two calls: `client.v2.parse` then `client.v2.extract` (schema in,
structured fields out). Let's prove it on a single document first.

In [ ]:
from ade_client import build_client, parse_and_extract
from invoice_schema import InvoiceExtractionSchema

client = build_client(S)
sample = files[0]
print(f'🔍 Parsing + extracting: {sample.split("/")[-1]} ...')
pr, er = parse_and_extract(client, sample, InvoiceExtractionSchema, S)

n_blocks = sum(len(p.children or []) for p in (pr.structure.children or []))
print(f'   ✅ {pr.metadata.page_count} page(s) parsed · {n_blocks} blocks found')

ex  = er.extraction
inv = ex.get('invoice_info', {}); co = ex.get('company_info', {}); tot = ex.get('totals_summary', {})
print('\n📋 Extracted fields:')
print(f'   Invoice #   : {inv.get("invoice_number")}')
print(f'   Date        : {inv.get("invoice_date")}')
print(f'   Supplier    : {co.get("supplier_name")}')
print(f'   Total due   : {tot.get("total_due")} {tot.get("currency") or ""}')
print(f'   Line items  : {len(ex.get("line_items", []))}')

## 4  ·  Connect to Snowflake

Opens **one** connection for the whole run. The first time, a browser window
pops up for SSO sign-in (it caches after that).

In [ ]:
from sf_loader import sf_connect, ensure_formats_and_stages

conn = sf_connect(S)                 # ← browser SSO login happens here (once)
ensure_formats_and_stages(S, conn)   # create ingest stage + file formats
print('✅ Connected to Snowflake · stages + file formats ready')

## 5  ·  Stream the batch into Snowflake  🚀

Documents are parsed + extracted **concurrently**, and each one's rows are staged
and `COPY`ed the moment it finishes — so rows land continuously. Watch the
**progress bar** (docs/s, rows→Snowflake) and the **Concurrency** line in the
summary: that's the speed story.

*Tip: open Snowsight and run `SELECT COUNT(*) FROM INVOICES_MAIN;` while this runs.*

In [ ]:
from pipeline import run_streaming

metrics = run_streaming(files, InvoiceExtractionSchema, S, conn=conn)
print(metrics.summary())

## 6  ·  Verify — the four tables + the originals stage

In [ ]:
from run_demo import verify_counts

verify_counts(S, conn=conn)

## 7  ·  Query the structured results

Plain SQL from here — headers joined to line items.

In [ ]:
from sf_loader import fq_table

query = f'''
SELECT m.supplier_name, m.invoice_number, m.total_due,
       COUNT(li.line_index) AS line_items
FROM {fq_table(S, S.table_main)} m
LEFT JOIN {fq_table(S, S.table_lines)} li USING (invoice_uuid)
GROUP BY 1, 2, 3
ORDER BY m.total_due DESC
'''
cur = conn.cursor(); cur.execute(query)
print(f'{"SUPPLIER":<34} {"INVOICE #":<14} {"TOTAL":>12}  LINES')
print('-' * 72)
for supplier, number, total, lines in cur.fetchall():
    print(f'{(supplier or "")[:33]:<34} {str(number):<14} {float(total):>12,.2f}  {lines}')

## Done

Close the connection. To scale up, point `gather_files` at a folder of your own PDFs — the pattern streams the same way whether it's 4 documents or 4,000.

In [ ]:
conn.close()
print('✅ Done — connection closed.')